# Datacenter Load Profile Generator

Creates hourly load profiles for datacenters based on Swedish/Nordic conditions.

**Key characteristics:**
- Baseload: ~90% load factor, 24/7/365 operations
- Counter-seasonal: Peak in summer (cooling), flat in winter
- No weekly variation: Operates identically all days
- Dual hourly patterns: Winter = flat, Summer = temperature-following

**Source:** See `README.md` in this folder for full methodology and sources.

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Configuration
YEAR = 2024
IS_LEAP_YEAR = (YEAR % 4 == 0 and YEAR % 100 != 0) or (YEAR % 400 == 0)
HOURS_IN_YEAR = 8784 if IS_LEAP_YEAR else 8760

# Output paths - save to parent directory (generator/input/load_profiles/)
# This matches the pattern used by other profile notebooks
DATA_PATH = Path('.')
OUTPUT_DIR = DATA_PATH.parent  # ../  = generator/input/load_profiles/
OUTPUT_CSV = OUTPUT_DIR / f'profile_datacenters_{YEAR}.csv'
OUTPUT_JSON = OUTPUT_DIR / f'profile_datacenters_patterns.json'

print(f"Year: {YEAR} ({'leap year' if IS_LEAP_YEAR else 'regular year'})")
print(f"Hours in year: {HOURS_IN_YEAR}")
print(f"Output CSV: {OUTPUT_CSV.resolve()}")
print(f"Output JSON: {OUTPUT_JSON.resolve()}")

## 2. Define Pattern Components

Values from README based on Swedish datacenter empirical data and industry research.

In [ ]:
# Monthly multipliers (sum = 12.00)
# Peak in July (cooling demand), minimum in winter (free cooling)
# Based on empirical data: ~3.3x summer/winter ratio with abrupt transitions
MONTHLY_MULT = {
    1: 0.65,   # January - Full free cooling (baseline)
    2: 0.65,   # February - Full free cooling
    3: 0.65,   # March - Full free cooling
    4: 0.65,   # April - Full free cooling
    5: 0.65,   # May - Full free cooling
    6: 2.00,   # June - Abrupt start of compressor cooling
    7: 2.15,   # July - Peak month
    8: 2.00,   # August - Heavy compressor use
    9: 0.65,   # September - Abrupt return to free cooling
    10: 0.65,  # October - Full free cooling
    11: 0.65,  # November - Full free cooling
    12: 0.65,  # December - Full free cooling
}

# Verify sum
monthly_sum = sum(MONTHLY_MULT.values())
print(f"Monthly multipliers sum: {monthly_sum:.2f} (expected: 12.00)")
assert abs(monthly_sum - 12.00) < 0.02, "Monthly multipliers must sum to 12.00"

# Show summer/winter ratio
winter_avg = MONTHLY_MULT[1]
summer_peak = MONTHLY_MULT[7]
print(f"Summer/Winter ratio: {summer_peak/winter_avg:.2f}x (target: ~3.3x)")

In [ ]:
# Weekday multipliers - NO variation for datacenters (24/7/365 operation)
WEEKDAY_MULT = {
    0: 1.00,  # Monday
    1: 1.00,  # Tuesday
    2: 1.00,  # Wednesday
    3: 1.00,  # Thursday
    4: 1.00,  # Friday
    5: 1.00,  # Saturday
    6: 1.00,  # Sunday
}

weekday_sum = sum(WEEKDAY_MULT.values())
print(f"Weekday multipliers sum: {weekday_sum:.2f} (expected: 7.00)")

In [ ]:
# Winter hourly pattern (Oct-May): COMPLETELY FLAT
# Full free cooling means zero variation - pure IT baseload
HOURLY_WINTER = {h: 1.00 for h in range(24)}

winter_sum = sum(HOURLY_WINTER.values())
print(f"Winter hourly sum: {winter_sum:.2f} (expected: 24.00)")
print(f"Winter peak/min ratio: {max(HOURLY_WINTER.values())/min(HOURLY_WINTER.values()):.3f} (completely flat)")

In [ ]:
# Summer hourly pattern (Jun-Aug): Large temperature-following swing
# Peak at 13:00-14:00 (warmest outdoor temps), minimum at 03:00-04:00
# Target: Peak/min ratio ~2.5 (daily swing is ~50-60% of max)
HOURLY_SUMMER = {
    0: 0.70,   # Night - minimal compressors
    1: 0.67,
    2: 0.65,   # Approaching minimum
    3: 0.65,   # Minimum - coolest outdoor temp
    4: 0.65,   # Minimum
    5: 0.70,   # Pre-dawn
    6: 0.80,   # Dawn, temps rising
    7: 0.92,   # Morning ramp
    8: 1.05,   # Compressors ramping up
    9: 1.20,   # Warming rapidly
    10: 1.35,  # Late morning
    11: 1.48,  # Approaching peak
    12: 1.56,  # Solar noon
    13: 1.60,  # Peak - warmest hour
    14: 1.56,  # Afternoon peak
    15: 1.48,  # Afternoon decline starts
    16: 1.35,  # Cooling begins
    17: 1.20,  # Evening transition
    18: 1.05,  # Compressors reducing
    19: 0.92,  # Evening cooling
    20: 0.82,  # Night transition
    21: 0.76,  # Night mode
    22: 0.73,  # Night mode
    23: 0.70,  # Night mode
}

summer_sum = sum(HOURLY_SUMMER.values())
print(f"Summer hourly sum: {summer_sum:.2f} (expected: 24.00)")

# Normalize to exactly 24.00
summer_factor = 24.00 / summer_sum
HOURLY_SUMMER = {h: v * summer_factor for h, v in HOURLY_SUMMER.items()}
print(f"Summer hourly sum (normalized): {sum(HOURLY_SUMMER.values()):.2f}")
print(f"Summer peak/min ratio: {max(HOURLY_SUMMER.values())/min(HOURLY_SUMMER.values()):.2f} (target: ~2.5)")

In [ ]:
# Define which months use summer vs winter pattern
# Summer: June (6) through August (8) - core compressor cooling period (abrupt start/stop)
# Winter: September (9) through May (5) - free cooling period
SUMMER_MONTHS = [6, 7, 8]  # Only 3 months - abrupt transition
WINTER_MONTHS = [1, 2, 3, 4, 5, 9, 10, 11, 12]  # All other months

print(f"Summer months (compressor cooling): {SUMMER_MONTHS}")
print(f"Winter months (free cooling): {WINTER_MONTHS}")
print(f"Note: Abrupt transition - summer pattern only in Jun-Aug")

## 3. Generate Profile

In [ ]:
# Generate timestamp index for the year
timestamps = pd.date_range(
    start=f'{YEAR}-01-01 00:00:00',
    end=f'{YEAR}-12-31 23:00:00',
    freq='h'
)

print(f"Generated {len(timestamps)} timestamps")
print(f"First: {timestamps[0]}")
print(f"Last: {timestamps[-1]}")

In [ ]:
def get_hourly_multiplier(month: int, hour: int) -> float:
    """Get hourly multiplier based on season (summer vs winter pattern)."""
    if month in SUMMER_MONTHS:
        return HOURLY_SUMMER[hour]
    else:
        return HOURLY_WINTER[hour]

# Generate raw profile values
values = []
for ts in timestamps:
    month = ts.month
    hour = ts.hour
    dow = ts.dayofweek  # 0=Monday, 6=Sunday
    
    # Combine multipliers
    hourly_mult = get_hourly_multiplier(month, hour)
    monthly_mult = MONTHLY_MULT[month]
    weekday_mult = WEEKDAY_MULT[dow]  # Always 1.0 for datacenters
    
    raw_value = hourly_mult * monthly_mult * weekday_mult
    values.append(raw_value)

# Create DataFrame
df = pd.DataFrame({
    'timestamp': timestamps,
    'raw_value': values
})

print(f"Generated {len(df)} hourly values")
print(f"\nRaw value stats:")
print(df['raw_value'].describe())

In [ ]:
# Normalize so values sum to 1.0
total = df['raw_value'].sum()
df['value'] = df['raw_value'] / total

print(f"Normalized profile:")
print(f"Sum: {df['value'].sum():.10f} (should be 1.0)")
print(f"Min: {df['value'].min():.10f}")
print(f"Max: {df['value'].max():.10f}")
print(f"Mean: {df['value'].mean():.10f}")

## 4. Validate

In [ ]:
# Validation checks
print("=== VALIDATION ===")

# 1. Sum equals 1.0
total_sum = df['value'].sum()
check1 = abs(total_sum - 1.0) < 1e-9
print(f"1. Sum equals 1.0: {'PASS' if check1 else 'FAIL'} (sum = {total_sum:.10f})")

# 2. All values positive
check2 = (df['value'] > 0).all()
print(f"2. All values positive: {'PASS' if check2 else 'FAIL'}")

# 3. July > January (seasonal pattern) - expect ~3x ratio
df['month'] = df['timestamp'].dt.month
july_avg = df[df['month'] == 7]['value'].mean()
jan_avg = df[df['month'] == 1]['value'].mean()
check3 = july_avg > jan_avg * 2.5  # Must be at least 2.5x
print(f"3. July > 2.5x January: {'PASS' if check3 else 'FAIL'} (July: {july_avg:.6f}, Jan: {jan_avg:.6f}, ratio: {july_avg/jan_avg:.2f}x)")

# 4. Hour 13 > Hour 3 in summer (daily pattern) - expect ~2.5x ratio
df['hour'] = df['timestamp'].dt.hour
summer_df = df[df['month'].isin(SUMMER_MONTHS)]
hour13_avg = summer_df[summer_df['hour'] == 13]['value'].mean()
hour3_avg = summer_df[summer_df['hour'] == 3]['value'].mean()
check4 = hour13_avg > hour3_avg * 2.0  # Must be at least 2x
print(f"4. Summer hour 13 > 2x hour 3: {'PASS' if check4 else 'FAIL'} (h13: {hour13_avg:.6f}, h3: {hour3_avg:.6f}, ratio: {hour13_avg/hour3_avg:.2f}x)")

# 5. Winter is completely flat
winter_df = df[df['month'].isin(WINTER_MONTHS)]
winter_by_hour = winter_df.groupby('hour')['value'].mean()
winter_ratio = winter_by_hour.max() / winter_by_hour.min()
check5 = winter_ratio < 1.001  # Essentially 1.0 (completely flat)
print(f"5. Winter is completely flat: {'PASS' if check5 else 'FAIL'} (peak/min ratio: {winter_ratio:.4f})")

# 6. No weekday variation
df['weekday'] = df['timestamp'].dt.dayofweek
weekday_avg = df.groupby('weekday')['value'].mean()
weekday_ratio = weekday_avg.max() / weekday_avg.min()
check6 = abs(weekday_ratio - 1.0) < 0.01  # Less than 1% variation
print(f"6. No weekday variation: {'PASS' if check6 else 'FAIL'} (max/min ratio: {weekday_ratio:.4f})")

# Overall
all_passed = all([check1, check2, check3, check4, check5, check6])
print(f"\n{'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")

## 5. Visualize

In [ ]:
# Annual profile - daily averages
df['date'] = df['timestamp'].dt.date
daily_avg = df.groupby('date')['value'].sum()  # Sum per day

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_avg.index, daily_avg.values, linewidth=0.5, alpha=0.8)
ax.fill_between(daily_avg.index, daily_avg.values, alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Daily Load (fraction of annual)')
ax.set_title(f'Datacenter Load Profile - {YEAR} (Daily Totals)')
ax.grid(True, alpha=0.3)

# Mark summer period
summer_start = pd.Timestamp(f'{YEAR}-05-01')
summer_end = pd.Timestamp(f'{YEAR}-09-30')
ax.axvspan(summer_start, summer_end, alpha=0.1, color='red', label='Summer (compressor cooling)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Monthly averages
monthly_avg = df.groupby('month')['value'].sum()
monthly_avg = monthly_avg / monthly_avg.sum() * 12  # Normalize to sum=12

fig, ax = plt.subplots(figsize=(10, 5))
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
colors = ['blue' if m in WINTER_MONTHS else 'red' for m in range(1, 13)]
bars = ax.bar(months, monthly_avg.values, color=colors, alpha=0.7)

ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Average (1.0)')
ax.set_xlabel('Month')
ax.set_ylabel('Monthly Multiplier')
ax.set_title('Datacenter Monthly Load Pattern')
ax.legend(['Average', 'Winter (free cooling)', 'Summer (compressor)'])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, monthly_avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Sample days: Winter vs Summer
# Pick a day in January and a day in July
winter_day = df[df['timestamp'].dt.date == pd.Timestamp(f'{YEAR}-01-15').date()]
summer_day = df[df['timestamp'].dt.date == pd.Timestamp(f'{YEAR}-07-15').date()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Winter day
ax = axes[0]
ax.plot(winter_day['hour'], winter_day['value'] * 1e6, marker='o', color='blue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Load (×10⁻⁶)')
ax.set_title(f'Winter Day (Jan 15, {YEAR}) - Flat Profile')
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24, 2))

# Summer day
ax = axes[1]
ax.plot(summer_day['hour'], summer_day['value'] * 1e6, marker='o', color='red')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Load (×10⁻⁶)')
ax.set_title(f'Summer Day (Jul 15, {YEAR}) - Temperature Following')
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24, 2))

# Set same y-axis range for comparison
ymin = min(winter_day['value'].min(), summer_day['value'].min()) * 1e6 * 0.9
ymax = max(winter_day['value'].max(), summer_day['value'].max()) * 1e6 * 1.1
axes[0].set_ylim(ymin, ymax)
axes[1].set_ylim(ymin, ymax)

plt.tight_layout()
plt.show()

In [ ]:
# Hourly pattern comparison: Winter vs Summer average
winter_hourly = df[df['month'].isin(WINTER_MONTHS)].groupby('hour')['value'].mean()
summer_hourly = df[df['month'].isin(SUMMER_MONTHS)].groupby('hour')['value'].mean()

# Normalize to make comparison easier
winter_hourly_norm = winter_hourly / winter_hourly.mean()
summer_hourly_norm = summer_hourly / summer_hourly.mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(24), winter_hourly_norm, marker='o', label='Winter (flat)', color='blue')
ax.plot(range(24), summer_hourly_norm, marker='s', label='Summer (temperature-following)', color='red')
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Normalized Load')
ax.set_title('Datacenter Hourly Patterns: Winter vs Summer')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
# Prepare export DataFrame
df_export = pd.DataFrame({
    'hour': range(len(df)),
    'value': df['value'].values
})

print(f"Export DataFrame:")
print(f"  Rows: {len(df_export)}")
print(f"  Sum: {df_export['value'].sum():.10f}")
print(f"\nFirst 5 rows:")
print(df_export.head())
print(f"\nLast 5 rows:")
print(df_export.tail())

In [ ]:
# Save CSV
df_export.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(f"Size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

In [ ]:
# Prepare and save JSON patterns
patterns = {
    'hourly_winter': [HOURLY_WINTER[h] for h in range(24)],
    'hourly_summer': [HOURLY_SUMMER[h] for h in range(24)],
    'weekday': [WEEKDAY_MULT[d] for d in range(7)],
    'monthly': [MONTHLY_MULT[m] for m in range(1, 13)],
    'summer_months': SUMMER_MONTHS,
    'winter_months': WINTER_MONTHS,
    'source_year': YEAR,
    'description': 'Swedish datacenter load profile with temperature-following cooling pattern',
    'notes': {
        'hourly_winter': 'Completely flat pattern for free cooling months (Sep-May)',
        'hourly_summer': 'Large temperature-following swing for compressor cooling months (Jun-Aug)',
        'weekday': 'All 1.0 - no weekly variation (24/7 operation)',
        'monthly': 'Peak in July (1.90), minimum in winter (0.58) - ~3.3x ratio',
        'summer_peak_hour': 13,
        'summer_peak_min_ratio': round(max(HOURLY_SUMMER.values()) / min(HOURLY_SUMMER.values()), 2),
        'seasonal_ratio': round(MONTHLY_MULT[7] / MONTHLY_MULT[1], 2),
        'transition': 'Abrupt - summer pattern only Jun-Aug',
    },
    'data_source': 'Swedish datacenter empirical data (2024) and industry research - see README.md'
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(patterns, f, indent=2)

print(f"Saved: {OUTPUT_JSON}")
print(f"Size: {OUTPUT_JSON.stat().st_size / 1024:.1f} KB")

In [ ]:
# Summary
print("="*60)
print("DATACENTER PROFILE GENERATION COMPLETE")
print("="*60)
print(f"\n1. CSV Profile: {OUTPUT_CSV}")
print(f"   - {len(df_export)} hourly values")
print(f"   - Sum = 1.0 (normalized)")
print(f"   - Ready for use with extend_from_pattern()")

print(f"\n2. JSON Patterns: {OUTPUT_JSON}")
print(f"   - Winter hourly pattern (flat)")
print(f"   - Summer hourly pattern (temperature-following)")
print(f"   - Monthly multipliers (summer peak)")
print(f"   - Weekday multipliers (all 1.0)")

print(f"\n3. Key Characteristics:")
print(f"   - Seasonal amplitude: July/Jan = {july_avg/jan_avg:.2f}")
print(f"   - Summer daily peak/min: {hour13_avg/hour3_avg:.2f}")
print(f"   - Winter daily variation: {winter_ratio:.3f}")
print(f"   - Weekday variation: {weekday_ratio:.4f} (essentially none)")